# Olist Brazilian E-Commerce — Notebook 1: Data Cleaning & Preparation

**Dataset:** Olist Brazilian E-Commerce Public Dataset (Kaggle) — 99,441 orders placed on
Brazil's largest department store marketplace between September 2016 and August 2018.

**Tools:** Python · pandas · NumPy · Matplotlib · Seaborn

---

## What this notebook does

This is the **data preparation** half of the project. It takes seven raw CSV exports from
Olist's operational systems and turns them into a small set of clean, validated,
analysis-ready tables.

Nothing here produces a business insight — that happens in `02_eda_analysis.ipynb`.
Keeping the two apart means the slow, one-off cleaning work runs once, and everything
downstream starts from a single trusted source of truth shared by the Python EDA, the SQL
queries and the Power BI dashboard.

## Pipeline

| Step | Section | Output |
|------|---------|--------|
| 1 | Load the seven raw tables | Raw dataframes |
| 2 | Profile every table | Shape, duplicates, nulls, dtypes |
| 3 | Clean each table individually | `*_clean` dataframes |
| 4 | Validate join keys, then join | `master_orders`, `master_products` |
| 5 | Export to `/clean` | CSVs consumed by notebook 2, SQL and Power BI |

## Guiding principles

- **Profile before cleaning.** Every decision below is made against a number that was
  measured first, not assumed.
- **Document every judgement call.** Wherever rows are dropped or values imputed, the
  markdown above the code says how many and why.
- **Assert, don't hope.** Each cleaning function ends with assertions, so the notebook
  fails loudly the moment an assumption breaks.

## 0 · Environment setup

Imports, warning suppression, and the global plotting style used across both notebooks
(`seaborn-v0_8-whitegrid` with the `husl` palette), so every chart in the project shares
one visual identity.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import datetime as dt

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

## 1 · Loading the raw data

Olist publishes the data as separate CSVs modelled like a real relational database rather
than as one flat file. Seven of them are needed for this project:

| Table | Grain | What it holds |
|-------|-------|---------------|
| `orders` | one row per order | status and the five lifecycle timestamps |
| `customers` | one row per order's customer record | city, state, and the `customer_unique_id` that links repeat buyers |
| `order_items` | one row per item within an order | product, seller, price, freight |
| `payments` | one row per payment record | payment type, instalments, value |
| `reviews` | one row per review submission | 1–5 score, optional title and comment |
| `products` | one row per product | category plus size and weight attributes |
| `category` | one row per category | Portuguese → English category translation |

`orders` is the hub of this schema: every other table joins to it directly or through one
hop.

In [ ]:
orders = pd.read_csv('/content/olist_orders_dataset.csv')
customers = pd.read_csv('/content/olist_customers_dataset.csv')
order_items = pd.read_csv('/content/olist_order_items_dataset.csv')
payments = pd.read_csv('/content/olist_order_payments_dataset.csv')
reviews = pd.read_csv('/content/olist_order_reviews_dataset.csv')
products = pd.read_csv('/content/olist_products_dataset.csv')
category = pd.read_csv('/content/product_category_name_translation.csv')


## 2 · Profiling every table before touching it

Before any cleaning decision can be justified, the same four questions need answering for
all seven tables: how big is it, are there duplicate rows, where are the nulls, and are
the dtypes what they should be.

`profile_table()` answers all four in one pass and is run over every table, so the
decisions in Section 3 rest on measured numbers rather than assumptions.

In [ ]:
def profile_table(df, name):
  print(f"{'='*50}")
  print(f"TABLE: {name}")
  print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
  print(f"Duplicates: {df.duplicated().sum():,}")
  print(f"\n Nulls per column: ")
  nulls = df.isnull().sum()
  for col, n in nulls[nulls>0].items():
    print(f"   {col}: {n:,} ({n/len(df)*100:.1f}%)")
  if nulls.sum() == 0:
    print("  None - clean!")
  print(f"\n Dtypes:")
  for col, dtype in df.dtypes.items():
    print(f"  {col}: {dtype}")
  print()

# Calling profiling function for every table
for df, name in [(orders,'Orders'), (customers,'Customers'), (order_items,'Order Items'), (payments,'Payments'), (reviews,'Reviews'), (products,"Products"), (category,'Category')]:
  profile_table(df, name)


TABLE: Orders
Shape: 99,441 rows x 8 columns
Duplicates: 0

 Nulls per column: 
   order_approved_at: 160 (0.2%)
   order_delivered_carrier_date: 1,783 (1.8%)
   order_delivered_customer_date: 2,965 (3.0%)

 Dtypes:
  order_id: object
  customer_id: object
  order_status: object
  order_purchase_timestamp: object
  order_approved_at: object
  order_delivered_carrier_date: object
  order_delivered_customer_date: object
  order_estimated_delivery_date: object

TABLE: Customers
Shape: 99,441 rows x 5 columns
Duplicates: 0

 Nulls per column: 
  None - clean!

 Dtypes:
  customer_id: object
  customer_unique_id: object
  customer_zip_code_prefix: int64
  customer_city: object
  customer_state: object

TABLE: Order Items
Shape: 112,650 rows x 7 columns
Duplicates: 0

 Nulls per column: 
  None - clean!

 Dtypes:
  order_id: object
  order_item_id: int64
  product_id: object
  seller_id: object
  shipping_limit_date: object
  price: float64
  freight_value: float64

TABLE: Payments
Shape: 10

### What the profile revealed

Five things stand out, and each one drives a decision in the next section:

- **`orders` timestamps are strings, not datetimes.** All five lifecycle columns load as
  `object`, so nothing time-based works until they are converted.
- **`orders` has three columns with nulls** — `order_approved_at` (160),
  `order_delivered_carrier_date` (1,783) and `order_delivered_customer_date` (2,965).
  These are not errors: an order that was cancelled or is still in transit legitimately
  has no delivery date.
- **`reviews` is mostly null by design** — 88.3% of reviews have no title and 58.7% no
  comment. A missing comment means the customer chose not to write one, so these nulls are
  kept, not filled.
- **`products` nulls come in two sizes** — 610 rows missing category and description
  metadata, and just 2 rows missing physical dimensions. Different causes, different
  treatment.
- **No exact duplicate rows anywhere.** Row-level duplication is not a problem in this
  dataset — but key-level duplication turns out to be, as Section 3.4 shows.

## 3 · Cleaning each table

Each table gets its own function that takes the raw dataframe, applies the decisions
documented above it, ends with assertions, and returns a clean copy. The raw dataframes
are never mutated in place — `df.copy()` at the top of every function keeps the originals
available for re-checking.

### 3.1 · Orders

Five decisions, in order:

1. **Convert the five timestamp columns to datetime** — nothing else works until this
   happens.
2. **Keep only `delivered` orders.** This project measures realised revenue, and
   cancelled, unavailable or in-transit orders have not generated any. Of 99,441 orders,
   96,478 are delivered.
3. **Drop the 8 delivered orders with no actual delivery date** — a delivered order without
   a delivery timestamp is self-contradictory, and there are too few to salvage.
4. **Fill the remaining null `order_approved_at` with the purchase timestamp.** These are
   delivered orders with no approval record; purchase time is the closest defensible proxy
   and it affects only a handful of rows.
5. **Engineer the time features the analysis needs** — `delivery_days`, `is_late`, and the
   purchase month/year/day-name/hour columns that drive the seasonality and day-hour charts
   in notebook 2.

The two assertions at the end enforce the contract this table must satisfy: one row per
order, and no unexpected nulls left anywhere.

In [ ]:
def clean_orders(df):
  df = df.copy()

  # Convert all timestamp columns to datetime
  ts_cols = ['order_purchase_timestamp','order_approved_at', 'order_delivered_carrier_date','order_delivered_customer_date', 'order_estimated_delivery_date']
  for col in ts_cols:
    df[col] = pd.to_datetime(df[col])

  # Keep only delivered orders for revenue analysis
  print(f"Order Statuses: {df.order_status.value_counts().to_dict()}")
  df = df[df.order_status == 'delivered'].copy()
  print(f"After filtering to delivered: {len(df):,} rows")

  # Drop rows missing actual delivery date
  before = len(df)
  df = df.dropna(subset = ['order_delivered_customer_date'])
  print(f"Dropped {before - len(df):,} rows with null delivery date")

  # Fill null order_approved_at with purchase timestamp (14 rows)
  # These are delivered orders with no approval record — purchase time is best proxy
  df['order_approved_at'] = df['order_approved_at'].fillna(df['order_purchase_timestamp'])

  # Drop the 1 row with null carrier date — too little data to impute reliably
  df = df.dropna(subset= ['order_delivered_carrier_date'])

  # Create useful derived columns
  df['delivery_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days
  df['is_late'] = (df['order_delivered_customer_date'] > df['order_estimated_delivery_date'])
  df['purchase_month'] = df['order_purchase_timestamp'].dt.to_period("M")
  df['purchase_year'] = df['order_purchase_timestamp'].dt.year
  df['purchase_day'] = df['order_purchase_timestamp'].dt.day_name()
  df['purchase_hour'] = df['order_purchase_timestamp'].dt.hour

  # Final validation

  assert df.order_id.duplicated().sum() == 0, "Duplicate order ids found!"
  assert df.isnull().sum().sum() == 0, f"Unexpected nulls remain:\n {df.isnull.sum()[df.isnull().sum() > 0]}"

  return df

orders_clean = clean_orders(orders)
print(f"\norders_clean: {orders_clean.shape}")



Order Statuses: {'delivered': 96478, 'shipped': 1107, 'canceled': 625, 'unavailable': 609, 'invoiced': 314, 'processing': 301, 'created': 5, 'approved': 2}
After filtering to delivered: 96,478 rows
Dropped 8 rows with null delivery date

orders_clean: (96469, 14)


**Result:** 96,469 clean delivered orders across 14 columns — 2,972 raw orders (3.0%)
removed, every one of them deliberately.

### 3.2 · Order items

`order_items` sits at item grain — one row per product line — while most of the analysis
needs order grain. This function therefore returns **two** dataframes:

- **`items_clean`** — item level, kept for the product-category analysis where each line
  matters.
- **`orders_revenue`** — aggregated to one row per order, carrying `revenue` (sum of
  prices), `freight`, `item_count` and `total_revenue`.

Zero-price rows are removed first. There are none, but the check documents that this was
verified rather than assumed. `item_total = price + freight_value` is added at item level
so shipping cost travels with the line it belongs to.

In [ ]:
def clean_order_items(df):
  df = df.copy()

  # Basic validation
  print(f"Nulls: {df.isnull().sum().sum()}")
  print(f"Zero Price Orders: {(df['price'] <=0).sum()}")

  # Remove zero price rows
  df = df[df['price']>0].copy()

  # Create total value per item
  df['item_total'] = df['price'] + df['freight_value']

  # Aggregate to Order level
  order_level = df.groupby('order_id').agg(
      revenue = ('price','sum'),
      freight = ('freight_value', 'sum'),
      item_count = ('order_item_id', 'count'),
  ).reset_index()

  order_level['total_revenue'] = order_level['revenue'] + order_level['freight']

  print(f"\n order_items_clean(order_level): {order_level.shape}")
  print(f"Revenue Range: R${order_level.revenue.min():.2f} - R${order_level.revenue.max():,.2f}")

  return df, order_level

items_clean, orders_revenue = clean_order_items(order_items)



Nulls: 0
Zero Price Orders: 0

 order_items_clean(order_level): (98666, 5)
Revenue Range: R$0.85 - R$13,440.00


**Result:** 98,666 orders with revenue attached, ranging from R\$0.85 to R\$13,440.00.

### 3.3 · Customers

Two things happen here, and the second is the single most important detail in the whole
notebook.

**City names are normalised.** Brazilian city names arrive with inconsistent case, stray
whitespace and accented characters, so `são paulo`, `Sao Paulo ` and `SAO PAULO` would
otherwise count as three different cities. Stripping, lowercasing and NFKD-normalising to
ASCII collapses them into one.

**`customer_id` is not a customer.** In this dataset `customer_id` is generated fresh for
every order — the same real person receives a different `customer_id` each time they buy.
The column that identifies an actual person across orders is `customer_unique_id`. The
print statements below make the gap visible: 99,441 `customer_id` values against 96,096
`customer_unique_id` values.

Every customer-level metric in notebook 2 — repeat rate, LTV, cohort retention, RFM — is
built on `customer_unique_id`. Using `customer_id` instead would make every customer look
like a one-time buyer and force the repeat purchase rate to 0%.

The state assertion is a light sanity check: Brazil has 26 states plus one federal
district, so anything above 27 means the column is corrupted.

In [ ]:
from numpy._core.defchararray import strip
def clean_customers(df):
  df = df.copy()

  # Standardize city names
  df['customer_city'] = (df['customer_city']
                         .str.strip()
                         .str.lower()
                         .str.normalize('NFKD')
                         .str.encode('ascii', errors='ignore')
                         .str.decode('utf-8'))

  # Validate state codes
  valid_states = df['customer_state'].unique()
  print(f"Unique states: {len(valid_states)} - {sorted(valid_states)}")
  assert len(valid_states) <= 27, "Unexpected number of states"

  print(f"\nUnique customers (uniue_id): {df['customer_unique_id'].nunique():,}")
  print(f"Unique customers (id):         {df['customer_id'].nunique():,}")
  print("Note: these differ because one person can have multiple orders")
  print(f"\ncustomers_clean: {df.shape}")

  return df

customers_clean = clean_customers(customers)




Unique states: 27 - ['AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RR', 'RS', 'SC', 'SE', 'SP', 'TO']

Unique customers (uniue_id): 96,096
Unique customers (id):         99,441
Note: these differ because one person can have multiple orders

customers_clean: (99441, 5)


### 3.4 · Reviews — investigating before cleaning

The profile showed no duplicate *rows* in `reviews`, but that only rules out rows which
are identical across all seven columns. Both key columns still need checking
independently, because a review table should have exactly one row per review and at most
one review per order.

The four cells below are that investigation, run before writing a single line of cleaning
code. Each one narrows the question further.

**Check 1 — is `review_id` unique?** If the same review id appears against more than one
order, something is structurally wrong. The second block counts how many ids repeat, and
the third pulls one repeating id out to inspect in full.

In [ ]:
# Check 1 — are review_ids unique?
print(f"Duplicate review_id count: {reviews['review_id'].duplicated().sum()}")

# Investigating further
# Step 1: how many times does each review_id appear?
review_id_counts = reviews['review_id'].value_counts()
print(review_id_counts.value_counts())

# Step 2: Look at duplicated review_id example
dup_review_id = reviews['review_id'].value_counts()[reviews['review_id'].value_counts() > 1].index[0]
example = reviews[reviews['review_id'] == dup_review_id]
print(example[['order_id','review_id','review_score',
               'review_comment_message','review_creation_date']].to_string())

Duplicate review_id count: 814
count
1    97621
2      764
3       25
Name: count, dtype: int64
                               order_id                         review_id  review_score                                                              review_comment_message review_creation_date
58393  cfdfd7862e532c4ec1ed1c6a1b56d320  4548534449b1f572e357211b90724f1b             1  quero devolver o produto comprei um veio outro;;;\r\nquero meu dinheiro de volta.   2018-08-18 00:00:00
92768  a2ac6dad85cf8af5b0afb510a240fe8c  4548534449b1f572e357211b90724f1b             1  quero devolver o produto comprei um veio outro;;;\r\nquero meu dinheiro de volta.   2018-08-18 00:00:00
95402  73a8ed1078e1d475d2b1fb7216a52e1a  4548534449b1f572e357211b90724f1b             1  quero devolver o produto comprei um veio outro;;;\r\nquero meu dinheiro de volta.   2018-08-18 00:00:00


**Check 2 — is `order_id` unique?** A different question from the one above: this asks
whether any single order carries more than one review.

In [ ]:
# Check 2 — how many order_ids appear more than once?
# Step 1 — check if order_id is unique in the reviews table
print(reviews['order_id'].duplicated().sum())
# If this prints anything > 0, there are duplicate order_ids

551


How many orders are affected, and how many reviews does each of them have?

In [ ]:
# Step 2 — how many orders have multiple reviews?
review_counts = reviews.groupby('order_id')['review_id'].count()
print(review_counts.value_counts())

review_id
1    98126
2      543
3        4
Name: count, dtype: int64


Finally, the same side-by-side inspection for a duplicated `order_id`. Note that here the
`review_id` values *differ* — which points to a completely different cause than Check 1.

In [ ]:
# Look at the actual duplicate rows side by side
dup_order_ids = reviews[reviews['order_id'].duplicated(keep=False)]['order_id'].unique()

# Pick one example and inspect it fully
example = reviews[reviews['order_id'] == dup_order_ids[0]]
print(example[['order_id','review_id','review_score',
               'review_comment_message','review_creation_date']].to_string())

                              order_id                         review_id  review_score review_comment_message review_creation_date
30    cf73e2cb1f4a9480ed70c154da3d954a  540e7bbb2d06cfb7f85f3a88ba7ac97f             5                    NaN  2018-01-18 00:00:00
3109  cf73e2cb1f4a9480ed70c154da3d954a  aa193e76d35950c4ae988237bb36ed2b             5                    NaN  2018-01-18 00:00:00


### Reviews Table — Two Separate Data Quality Issues Found

#### Issue 1 — Duplicated review_ids across different orders

Investigation revealed 764 review_ids appear twice and 25 appear three times — each
assigned to completely different order_ids with identical review text and scores. This is
a data entry error in the source dataset, not customer behaviour.

- **Decision:** drop all rows with non-unique review_ids.
- **Rationale:** cannot determine which order the review genuinely belongs to.
- **Rows affected:** ~1,578 (764×2 + 25×3)

#### Issue 2 — Same order submitted multiple times

543 orders have 2 review submissions, 4 orders have 3 submissions. Same order_id,
different review_ids, same date — customer submitted the form multiple times.

- **Decision:** keep the most recent submission per order_id.
- **Rows affected:** ~551

### Applying both decisions

`clean_reviews()` implements exactly the two decisions above, in that order — drop the
non-unique `review_id` rows first, then de-duplicate whatever `order_id` collisions
remain by keeping the most recent submission. Order matters: doing it the other way round
would risk keeping rows whose review id is untrustworthy.

It also returns two versions of the table:

- **`reviews_all`** — every valid review, used for score analysis.
- **`reviews_text`** — only reviews with a comment longer than 10 characters, reserved for
  the sentiment/NLP stage later in the project.

In [ ]:
def clean_reviews(df):
  df = df.copy()

  # COnvert timestamps
  df['review_creation_date'] = pd.to_datetime(df['review_creation_date'])
  df['review_answer_timestamp'] = pd.to_datetime(df['review_answer_timestamp'])

  # Validate review_score
  assert df['review_score'].between(1,5).all(), "Scored outside 1-5 range!"

  # Issue 1 — drop rows where review_id is duplicated across different orders
  # Same review_id + same text assigned to multiple orders = data entry error
  # Cannot determine which order the review truly belongs to — safest to drop
  before = len(df)
  dup_review_ids = df['review_id'].value_counts()
  dup_review_ids = dup_review_ids[dup_review_ids > 1].index
  df= df[~df['review_id'].isin(dup_review_ids)].copy()
  print(f"Dropped {before - len(df):,} rows with duplicate review_ids")

  # Issue 2 — after above, handle any remaining duplicate order_ids
  # (same order submitted review form twice — keep latest)
  before = len(df)
  df = (df.sort_values('review_creation_date', ascending=False).drop_duplicates(subset='order_id', keep='first').reset_index(drop=True))
  print(f"Dropped {before- len(df):,} duplicate review submissions(kept latest)")

  # Verify both issues are resolved
  assert df['review_id'].duplicated().sum() == 0, "review_id still has duplicates!"
  assert df['order_id'].duplicated().sum() == 0, "order_id still has dupicates!"

  # Split into two versions
  reviews_all = df.copy()
  reviews_text = df.dropna(subset=['review_comment_message'])
  reviews_text = reviews_text[reviews_text['review_comment_message'].str.len() > 10].copy()

  print(f"\nreviews_all: {len(reviews_all):,} rows")
  print(f"reviews_text: {len(reviews_text):,} rows")
  return reviews_all, reviews_text

reviews_clean, reviews_text_clean = clean_reviews(reviews)




Dropped 1,603 rows with duplicate review_ids
Dropped 91 duplicate review submissions(kept latest)

reviews_all: 97,530 rows
reviews_text: 36,848 rows


**Result:** 1,694 rows removed in total (1,603 + 91); 97,530 clean reviews remain, of
which 36,848 carry usable comment text.

### 3.5 · Products and payments

Two smaller tables, one function each.

**Products** — the 610 rows with no category are labelled `unknown` rather than dropped,
since those items still carry revenue that belongs in the totals. The Portuguese category
names are then joined to their English translations, and only `product_id` and
`category_english` are kept: the size and weight columns are not used anywhere in this
analysis.

**Payments** — a single order can have several payment rows (instalments, or a voucher
plus a card). Collapsing to one row per order needs two different rules: the **payment
type** comes from the first payment record (`payment_sequential == 1`), while the
**payment value** is summed across all records, so an order split across two methods is
not undercounted. Zero-value rows are dropped and zero-instalment values corrected to 1.

In [ ]:
def clean_products(df, translation_df):
  df= df.copy()
  # Fill null category with "unknown"
  df['product_category_name'] = df['product_category_name'].fillna('unknown')

  # Join English translation using the table product_category_name_translation
  df= df.merge(translation_df, on='product_category_name', how= 'left')
  df['category_english'] = df['product_category_name_english'].fillna('unknown')

  # Keep only what we need
  return df[['product_id','category_english']]

def clean_payments(df):
  df = df.copy()
  df = df[df['payment_value'] > 0].copy()
  # Fix zero installments
  df['payment_installments'] = df['payment_installments'].replace(0,1)

  # Get primary payment per order (sequential == 1)
  primary = df[df['payment_sequential'] == 1][['order_id','payment_type']]

  # Get total payemnt value per order
  total = df.groupby('order_id')['payment_value'].sum().reset_index()
  return primary.merge(total, on='order_id')

products_clean = clean_products(products, category)
payments_clean = clean_payments(payments)
print("All tables cleaned!!")



All tables cleaned!!


Quick confirmation of the shape `clean_payments()` returns before it is used in a join —
one row per order, with a type and a total.

In [ ]:
payments_clean.columns



Index(['order_id', 'payment_type', 'payment_value'], dtype='object')

---

## 4 · Joining the clean tables

Seven tables now need to become two analysis tables. The risk in any join is silent row
loss or silent row multiplication — a merge that quietly duplicates rows is the classic
way an analysis ends up reporting revenue that never existed.

### 4.1 · Validating join keys before joining

`validate_join_keys()` runs before every merge and reports four things for each side of
the join: nulls in the key, duplicates in the key, and how many key values are present in
one table only.

Read the output as a prediction: keys "in orders only" become nulls after a left join,
keys in the right table only get dropped. Knowing those numbers *before* merging means any
surprise in the row count afterwards is immediately explainable.

In [ ]:
def validate_join_keys(left_df, right_df, key, left_name, right_name):
  left_nulls = left_df[key].isnull().sum()
  right_nulls = right_df[key].isnull().sum()
  left_dupes = left_df[key].duplicated().sum()
  right_dupes = right_df[key].duplicated().sum()

  print(f"\nJoin Key validation: {key}")
  print(f"{left_name} - nulls: {left_nulls}, dupes: {left_dupes}")
  print(f"{right_name} - nulls: {right_nulls}, dupes: {right_dupes}")

  # Overlap Check
  left_keys = set(left_df[key].dropna())
  right_keys = set(right_df[key].dropna())
  in_left_only = len(left_keys - right_keys)
  in_right_only = len(right_keys - left_keys)
  in_both = len(left_keys & right_keys)

  print(f" Keys in both: {in_both}")
  print(f" In {left_name} only: {in_left_only:,} (will be nulls after left join)")
  print(f" In {right_name} only: {in_right_only:,} (will be dropped in left join)")

# Run before every join
validate_join_keys(orders_clean, orders_revenue, 'order_id', 'orders', 'order_items')
validate_join_keys(orders_clean, customers_clean, 'customer_id', 'orders', 'customers')
validate_join_keys(orders_clean, reviews_clean, 'order_id', 'orders', 'reviews')
validate_join_keys(orders_clean, payments_clean, 'order_id', 'orders', 'payments')





Join Key validation: order_id
orders - nulls: 0, dupes: 0
order_items - nulls: 0, dupes: 0
 Keys in both: 96469
 In orders only: 0 (will be nulls after left join)
 In order_items only: 2,197 (will be dropped in left join)

Join Key validation: customer_id
orders - nulls: 0, dupes: 0
customers - nulls: 0, dupes: 0
 Keys in both: 96469
 In orders only: 0 (will be nulls after left join)
 In customers only: 2,972 (will be dropped in left join)

Join Key validation: order_id
orders - nulls: 0, dupes: 0
reviews - nulls: 0, dupes: 0
 Keys in both: 94773
 In orders only: 1,696 (will be nulls after left join)
 In reviews only: 2,757 (will be dropped in left join)

Join Key validation: order_id
orders - nulls: 0, dupes: 0
payments - nulls: 0, dupes: 0
 Keys in both: 96390
 In orders only: 79 (will be nulls after left join)
 In payments only: 2,967 (will be dropped in left join)


### What the validation tells us

No nulls and no duplicate keys on either side of any of the four joins — the structural
precondition for a safe merge is met everywhere.

The rest is expected, and explained by the fact that `orders_clean` has already been
filtered down to delivered orders:

- **order_items** — 2,197 order ids exist only in the item table (the non-delivered orders
  that were filtered out); all 96,469 clean orders have items.
- **customers** — the same story: 2,972 customer records belong to filtered-out orders.
- **reviews** — 1,696 delivered orders have no review at all. That is real customer
  behaviour, not a data problem, so this join must be a **left** join or those orders would
  vanish from the revenue totals.
- **payments** — 79 delivered orders have no payment record; again a left join.

### 4.2 · Building `master_orders`

One row per delivered order, joined out to everything that describes it. The join type
differs by table, and each choice is deliberate:

| Joined table | Type | Why |
|---|---|---|
| `orders_revenue` | inner | an order with no items has no revenue and cannot be analysed |
| `customers_clean` | inner | every order must have a customer; no customer means a broken record |
| `reviews_clean` | **left** | 1,696 orders have no review — dropping them would bias revenue downward |
| `payments_clean` | **left** | 79 orders have no payment record, same reasoning |

Row counts are printed after every step, so unexpected loss or duplication is caught at
the moment it happens rather than three cells later. The final assertion guarantees the
promised grain: exactly one row per order.

In [ ]:
def build_master_orders(orders_clean, orders_revenue, customers_clean, reviews_clean, payments_clean):
  df = orders_clean.copy()
  n0 = len(df)
  print(f"Start: {n0:,} rows")

  # Join revenue (inner join - only keep orders that have items)
  df = df.merge(orders_revenue, how='inner', on='order_id')
  print(f"After joining revenue: {len(df):,} rows (dropped {n0-len(df):,} orders with no items)")

  # Join customers (inner join - need customers info)
  df = df.merge(customers_clean[['customer_id','customer_unique_id','customer_city','customer_state' ]], how='inner', on='customer_id')
  print(f"After joining customers: {len(df):,} rows")

  # Join reviews (Left join - not every order has a review)
  df = df.merge(reviews_clean[['order_id', 'review_score']], how='left', on='order_id')
  print(f"After joining reviews: {len(df):,} rows")
  print(f"  Orders with reviews: {df['review_score'].notna().sum():,}")
  print(f"  Orders without reviews: {df['review_score'].isna().sum():,}")

  # Join payments (Left join - keep all orders)
  df = df.merge(payments_clean,how='left', on='order_id')
  print(f"After joining payments: {len(df):,} rows")

  # Final sanity check
  assert df['order_id'].duplicated().sum() == 0, f"PROBLEM: {df['order_id'].duplicated().sum()} duplicate order_id after join!"
  print(f"\nFinal master orders: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
  print("No duplicate order_ids")
  return df

master_orders = build_master_orders(orders_clean,orders_revenue,customers_clean,reviews_clean,payments_clean)




Start: 96,469 rows
After joining revenue: 96,469 rows (dropped 0 orders with no items)
After joining customers: 96,469 rows
After joining reviews: 96,469 rows
  Orders with reviews: 94,773
  Orders without reviews: 1,696
After joining payments: 96,469 rows

Final master orders: 96,469 rows x 24 columns
No duplicate order_ids


**Result:** 96,469 rows × 24 columns, no duplicate order ids, and the row count unchanged
through all four joins — the grain held.

### 4.3 · Building `master_products`

Product analysis uses a different granularity (item-level, not order-level). Keeping it as a separate dataframe — merging it into master_orders would create multiple rows per order.

In [ ]:
master_products = (items_clean.
                   merge(products_clean,how='left', on='product_id').
                   merge(orders_clean[['order_id','purchase_month', 'purchase_year','purchase_day','purchase_hour']], how='left', on='order_id'))

# Check join quality
unmatched_products = master_products['category_english'].isna().sum()
unmatched_pct = unmatched_products/ len(master_products) * 100
print(f"Items with no matching product category: {unmatched_products:,} ({unmatched_pct:.2f}%)")
print(f"Items with no matching order date: {master_products['purchase_month'].isna().sum():,}")

assert len(master_products) == len(items_clean), f"Row count changed! items_clean: {len(items_clean):,}, master_products: {len(master_products):,}"

print(f"\n master_products: {master_products.shape}")

Items with no matching product category: 0 (0.00%)
Items with no matching order date: 2,462

 master_products: (112650, 13)


**Result:** 112,650 item rows, every one matched to a product category. The 2,462 rows
with no purchase month are items belonging to non-delivered orders, which were filtered
out of `orders_clean` — expected, and harmless, because all time-based product analysis in
notebook 2 runs on the delivered subset.

---

## 5 · Exporting the clean tables

Everything downstream reads from `/content/clean`, so all three tools work from one source
of truth:

- **Notebook 2 (`02_eda_analysis.ipynb`)** loads `master_orders`, `master_products` and
  `reviews_text_clean`.
- **SQL (DuckDB)** queries the same CSVs directly.
- **Power BI** loads the individual clean tables and rebuilds the relationships in its own
  model — which is why they are exported separately as well as pre-joined.

In [ ]:
orders_clean.to_csv("/content/clean/orders.csv", index=False)
reviews_clean.to_csv("/content/clean/reviews.csv", index=False)
payments_clean.to_csv("/content/clean/payments.csv", index= False)
customers_clean.to_csv("/content/clean/customers.csv", index=False)
products_clean.to_csv("/content/clean/products.csv", index=False)
items_clean.to_csv("/content/clean/order_items.csv", index=False)
master_orders.to_csv("/content/clean/master_orders.csv", index= False)
master_products.to_csv('/content/clean/master_products.csv', index=False)

print("All clean files exported ✓")
print("\nFor Power BI: load the individual clean tables and create relationships")
print("For Python analysis: use master_orders and master_products")

All clean files exported ✓

For Power BI: load the individual clean tables and create relationships
For Python analysis: use master_orders and master_products


The text-only review subset is exported separately: it is the input for the sentiment
analysis stage, not for the EDA.

In [ ]:
reviews_text_clean.to_csv("/content/clean/reviews_text_clean.csv", index=False)

---

## Summary — what this notebook produced

| Output | Grain | Rows |
|---|---|---|
| `master_orders.csv` | one row per delivered order | 96,469 |
| `master_products.csv` | one row per order item | 112,650 |
| `reviews_text_clean.csv` | reviews with usable comment text | 36,848 |
| `orders` · `customers` · `order_items` · `payments` · `products` · `reviews` | individual clean tables for Power BI | — |

### The four cleaning decisions worth remembering

1. **Filtered to delivered orders only** — this project measures realised revenue, not
   booked revenue.
2. **Used `customer_unique_id`, never `customer_id`,** for anything customer-level.
3. **Found and separated two distinct duplication problems** in the reviews table that a
   standard `duplicated()` check does not surface.
4. **Validated every join key before merging,** and chose inner vs left per join rather
   than defaulting to one.

**Next:** `02_eda_analysis.ipynb` — revenue, products, customers, geography, RFM
segmentation and review analysis.